# S3 Demo: Q-Learning, Live (Topic 3.1)
A tabular agent learns a 4×4 grid world in front of you. All local, all visible.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

In [ ]:
from course_utils.rl import GridWorld, train_q_learning, greedy_run, print_policy
env = GridWorld()          # S start · G goal(+1) · H holes(end)
env.render()

## Baseline: the untrained agent wanders

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
for ep in range(3):
    s, done, steps = env.reset(), False, 0
    while not done and steps < 20:
        s, r, done = env.step(int(rng.integers(4))); steps += 1
    print(f'episode {ep}: reward={r}, steps={steps}')

## The whole algorithm is one line
`Q[s,a] += lr * (r + γ·max Q[s'] − Q[s,a])`
new estimate ← old + step · (surprise). Everything else is bookkeeping.

In [ ]:
env = GridWorld()
Q, rewards = train_q_learning(env, episodes=2000)
from course_utils import viz
viz.plot_training(rewards)

## What it learned: the policy, as arrows

In [ ]:
print_policy(env, Q)
path, total, reached = greedy_run(env, Q)
print('greedy run:', ' -> '.join(map(str, path)))
print('total reward:', total, '| reached goal:', reached)

## Reward hacking, live
New board: a **bonus tile B** next to the start. Each time the agent is on B it gets **+0.1**, and the game goes on.
We meant "a small tip for passing by". What will the agent learn?

In [ ]:
from course_utils.rl import BONUS_MAP
env_b = GridWorld(BONUS_MAP, bonus=0.1)
env_b.render()
Q_b, rewards_b = train_q_learning(env_b, episodes=2000)
print_policy(env_b, Q_b)
path, total, reached = greedy_run(env_b, Q_b)
print('first steps:', ' -> '.join(map(str, path[:8])), '...')
print('total reward in 30 steps:', total, '| reached goal:', reached)

It found a loophole: it stands on B and walks into the wall. It never moves, so it collects +0.1 every step.
That pays more than finishing (+1 once). **Nothing is broken. The reward was wrong.**

## Fix the reward, not the agent
Make the bonus small enough that finishing is worth more.

In [ ]:
env_fix = GridWorld(BONUS_MAP, bonus=0.01)
Q_fix, _ = train_q_learning(env_fix, episodes=2000)
print_policy(env_fix, Q_fix)
path, total, reached = greedy_run(env_fix, Q_fix)
print('greedy run:', ' -> '.join(map(str, path)))
print('total reward:', total, '| reached goal:', reached)

**You just watched reward design change behavior.** That is both the power and the risk (→ CoastRunners, → Session 5 alignment).